# Description

W tym notatniku przeprowadzane są wszelkie eksperymenty, zarówno dla autoenkodera wariacyjnego i nie wariacyjnego, dla wszystkich członów funkcji straty, w wersji z douczaniem i bez (łącznie 12 eksperymentów)

# Imports

In [42]:
%load_ext autoreload
%autoreload 2
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import RichProgressBar
import yaml
import sys
import os
import tqdm
import wandb
import json

sys.path.append('../')  # Dodajemy katalog wyżej, żeby src był widoczny
from src.models.KlejdaGAE.GraphAutoencoder import GraphAutoencoder
from src.models.KlejdaGAE.VariationalGraphAutoencoder import VariationalGraphAutoencoder
from utils.FramsticksGraphDataset import FramsticksGraphDataset
from pyprojroot import here

current_dir = os.getcwd()
framspy_path = os.path.abspath(os.path.join(current_dir, '..', 'external', 'framspy'))
if framspy_path not in sys.path:
	sys.path.insert(0, framspy_path)
from FramsticksLib import FramsticksLib
from deap import tools, algorithms
import yaml
from src.deap.deap_setup import prepare_native_toolbox
from src.deap.constraints import is_feasible_fitness_criteria
from src.deap.save_and_load_results import save_genotypes_json
from utils.FramsticksPostProcessor import FramsticksPostProcessor
from utils.FramsticksGraphDataset import FramsticksGraphDataset
import numpy as np
import time

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Pre-processing

In [43]:
project_dir = here()
# Przygotowanie checkpointów nauczonych autoenkoderów
checkpoints_dir = project_dir / 'notebooks' / 'checkpoints' / 'final_checkpoints' / 'klejda'
checkpoint_gae = torch.load(checkpoints_dir / 'gae.ckpt')
checkpoint_vgae = torch.load(checkpoints_dir / 'vgae.ckpt')

In [44]:
# Przygotowanie konfiguracji dla gae
configs_dir = project_dir / 'configs'
config_gae_path = configs_dir / 'klejda_gae_config.yaml'
config_vgae_path = configs_dir / 'klejda_vgae_config.yaml'
with open(config_gae_path) as f:
    config_gae = yaml.safe_load(f)
with open(config_vgae_path) as f:
    config_vgae = yaml.safe_load(f)

In [54]:
# Implementacja nowego operatora mutacji
def autoencoder_mutate(gae, individual, sigma=1):
	framsticks_genotype = individual[0]

	x_matrix, a_matrix, _ = FramsticksGraphDataset.parse_f0_to_matrices(framsticks_genotype, evolution_config['max_numparts'])
	x_matrix.unsqueeze_(0)
	a_matrix.unsqueeze_(0)

	gae.eval()

	with torch.no_grad():
		z = gae.encode(x_matrix,a_matrix)

		noise = torch.randn_like(z) * sigma
		z_mutated = z + noise

		x_prime, a_prime = gae.decode(z_mutated)

		a_prime.squeeze_(0)
		x_prime.squeeze_(0)

		# 6. Konwersja: Wyjście GAE -> Framsticks
		_, new_framsticks_genotype, _ = postProcessingGaeResult.process(x_prime,a_prime)

		# 7. Zastąpienie starego genotypu nowym
		individual[0] = new_framsticks_genotype

		return individual,

In [55]:
# Przygotowanie środowiska Framsticks oraz DEAP
with open("../configs/final_evolution_config.yaml", 'r') as f:
	evolution_config = yaml.safe_load(f)

frams_lib = FramsticksLib(evolution_config['frams_path'], evolution_config['frams_lib'], evolution_config['sim_file'])

toolbox = prepare_native_toolbox(frams_lib, evolution_config)
# TODO: TO dodać przed uruchomieniem eksperymentu
# toolbox.register("mutate", autoencoder_mutate, gae)
pop = toolbox.population(n=evolution_config['pop_size'])
hof = tools.HallOfFame(evolution_config['hof_size'])

stats = tools.Statistics(lambda ind: ind.fitness.values)
filter_feasible = lambda func, criteria: func(list(filter(is_feasible_fitness_criteria, criteria)))
stats.register("min", lambda fit: filter_feasible(np.min, fit))
stats.register("avg", lambda fit: filter_feasible(np.mean, fit))
stats.register("max", lambda fit: filter_feasible(np.max, fit))

postProcessingGaeResult = FramsticksPostProcessor()

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data

Available objects: ['CheckpointEvent', 'Collision', 'CrCollision', 'Creature', 'CreatureSettings', 'CreatureSignals', 'CreatureSnapshot', 'Dictionary', 'ExpProperties', 'ExpState', 'ExtValue', 'File', 'FunctionReference', 'GenMan', 'GenManStats', 'GenePool', 'GenePools', 'Geno', 'GenoConverters', 'Genotype', 'Interface', 'Joint', 'Loader', 'Math', 'MechJoint', 'MechPart', 'MessageCatcher', 'Model', 'ModelGeometry', 'ModelSymmetry', 'Neuro', 'NeuroClass', 'NeuroClassLibrary', 'NeuroDef', 'NeuroSignals', 'NeuronsSimEnabled', 'ODE', 'Orient', 'Part', 'Population', 'Populations', 'Ref', 'Signal', 'SignalView', 'SimilMeasure', 'SimilMeasureDistribution', 'SimilMeasureGreedy', 'SimilMeasureHungarian', 'Simulator', 'SlaveSimulators', 'StopEvent', 'String', 'UserScripts', 'Vec

# Eksperymenty

## GAE

### Non-cyclic

In [56]:
gae_non_cyclic = GraphAutoencoder(config=config_gae).double()
gae_non_cyclic.load_state_dict(checkpoint_gae['state_dict'])
gae_non_cyclic.eval()
toolbox.register("mutate", autoencoder_mutate, gae_non_cyclic)

pop, log = algorithms.eaSimple(
	pop, toolbox,
	cxpb=evolution_config['p_xov'], mutpb=evolution_config['p_mut'],
	ngen=evolution_config['generations'], stats=stats, halloffame=hof, verbose=True
)

print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

gen	nevals	min  	avg  	max  
0  	120   	-0.01	-0.01	-0.01
1  	112   	-0.01	0.0338051	0.407039
2  	104   	-0.01	0.18678  	0.407039
3  	108   	0.225425	0.388349 	0.407039
4  	108   	0.407039	0.407039 	0.407039
5  	109   	0.407039	0.407039 	0.407039
6  	106   	0.407039	0.407039 	0.407039
7  	108   	0.407039	0.407039 	0.407039
8  	110   	0.407039	0.407039 	0.407039
9  	107   	0.407039	0.407039 	0.407039
10 	109   	0.407039	0.407039 	0.407039
11 	104   	0.407039	0.407039 	0.407039
12 	114   	0.407039	0.407039 	0.407039
13 	109   	0.407039	0.407039 	0.407039
14 	107   	0.407039	0.407039 	0.407039
15 	107   	0.407039	0.407039 	0.407039
16 	110   	0.407039	0.407039 	0.407039


Exception ignored in: <function ExtValue.__del__ at 0x000001A277DFFBA0>
Traceback (most recent call last):
  File "C:\Users\witek\PycharmProjects\Magisterka\external\framspy\frams_extvalue.py", line 40, in __del__
    c_api.extFree(self.__ptr)
                  ^^^^^^^^^^
  File "C:\Users\witek\PycharmProjects\Magisterka\external\framspy\frams_extvalue.py", line 212, in __getattr__
    return self.__dict__[key]
           ~~~~~~~~~~~~~^^^^^
KeyError: '_ExtValue__ptr'
Exception ignored in: <function ExtValue.__del__ at 0x000001A277DFFBA0>
Traceback (most recent call last):
  File "C:\Users\witek\PycharmProjects\Magisterka\external\framspy\frams_extvalue.py", line 40, in __del__
    c_api.extFree(self.__ptr)
                  ^^^^^^^^^^
  File "C:\Users\witek\PycharmProjects\Magisterka\external\framspy\frams_extvalue.py", line 212, in __getattr__
    return self.__dict__[key]
           ~~~~~~~~~~~~~^^^^^
KeyError: '_ExtValue__ptr'


17 	111   	0.407039	0.407039 	0.407039
18 	101   	0.407039	0.407039 	0.407039
19 	106   	0.407039	0.407039 	0.407039
20 	106   	0.407039	0.407039 	0.407039
21 	111   	0.407039	0.407039 	0.407039
22 	108   	0.407039	0.407039 	0.407039
23 	107   	0.407039	0.407039 	0.407039
24 	103   	0.407039	0.407039 	0.407039
25 	106   	0.407039	0.407039 	0.407039
26 	108   	0.407039	0.407039 	0.407039
27 	111   	0.407039	0.407039 	0.407039
28 	106   	0.407039	0.407039 	0.407039
29 	106   	0.407039	0.407039 	0.407039
30 	113   	0.407039	0.407039 	0.407039
31 	109   	0.407039	0.407039 	0.407039
32 	112   	0.407039	0.407039 	0.407039
33 	107   	0.407039	0.407039 	0.407039
34 	106   	0.407039	0.407039 	0.407039
35 	104   	0.407039	0.407039 	0.407039
36 	114   	0.407039	0.407039 	0.407039
37 	98    	0.407039	0.407039 	0.407039
38 	106   	0.407039	0.407039 	0.407039
39 	105   	0.407039	0.407039 	0.407039
40 	110   	0.407039	0.407039 	0.407039
41 	112   	0.407039	0.407039 	0.407039
42 	107   	0.407039	0.407

### Cyclic

## VGAE

### Non-cyclic

In [41]:
vgae_non_cyclic = VariationalGraphAutoencoder(config=config_vgae).double()
vgae_non_cyclic.load_state_dict(checkpoint_vgae['state_dict'])
vgae_non_cyclic.eval()
toolbox.register("mutate", autoencoder_mutate, vgae_non_cyclic)

pop, log = algorithms.eaSimple(
	pop, toolbox,
	cxpb=evolution_config['p_xov'], mutpb=evolution_config['p_mut'],
	ngen=evolution_config['generations'], stats=stats, halloffame=hof, verbose=True
)

print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

gen	nevals	min  	avg  	max  
0  	120   	-0.01	-0.01	-0.01
1  	112   	-0.01	-0.00413509	0.0427842
2  	100   	-0.01	0.0609979  	0.85406  
3  	109   	0.0427842	0.134123   	0.85406  
4  	107   	0.0427842	0.358689   	0.85406  
5  	107   	0.107268 	0.579329   	0.85406  
6  	105   	0.147824 	0.792657   	0.85406  
7  	111   	0.85406  	0.85406    	0.85406  
8  	110   	0.371239 	0.810167   	0.85406  
9  	107   	0.127316 	0.80215    	0.85406  
10 	108   	0.280549 	0.809944   	0.85406  
11 	109   	0.0570733	0.743784   	0.85406  
12 	110   	0.126926 	0.699555   	0.85406  
13 	111   	0.0716033	0.71233    	0.85406  
14 	103   	0.0852458	0.779533   	0.85406  
15 	112   	0.17232  	0.778311   	0.85406  
16 	112   	0.287413 	0.791099   	0.85406  
17 	112   	0.85406  	0.85406    	0.85406  
18 	110   	0.359093 	0.784679   	0.85406  
19 	114   	0.85406  	0.85406    	0.85406  
20 	107   	0.169194 	0.805141   	0.85406  
21 	113   	0.115218 	0.693205   	0.85406  
22 	108   	0.167816 	0.801272   	0.85406  
23 	

### Cyclic

# Results